# otro_pipe/02 — descomposicion EEMD + pronostico por componente (grilla, sin leakage)

Idea de la literatura de forecasting "decompose-ensemble": en vez de pronosticar `tn` de un
producto directo, se lo separa en sub-senales de distinta frecuencia con **EEMD** (Ensemble
Empirical Mode Decomposition), se pronostica cada sub-senal por separado, y se suman los
pronosticos para reconstruir `tn` (Wu, Huang & Chen, 2009; Lei & Zuo, 2009).

**Cambio de fondo vs. la version anterior**: la descomposicion EEMD es una transformacion
GLOBAL (no causal fila por fila como un lag) -- descomponer la serie completa (pasado +
futuro) y despues usar esos componentes para medir `wape_val`/`wape_test` es leakage real: el
algoritmo "ve" el valor que se supone hay que adivinar. La version anterior de este notebook
hacia eso. Ahora se corren **TRES descomposiciones independientes por combinacion**:

- `componentes_val` (corte en el ultimo mes de `meses_val`): sirve para medir `wape_val`
  (entrena con `meses_train`, valida en `meses_val`) sin que la descomposicion haya visto el
  target de ningun mes de validacion.
- `componentes_test` (corte en el ultimo mes de `meses_test`): sirve para medir `wape_test`
  (entrena con `meses_train`+`meses_val`, valida en `meses_test`) por la misma razon.
- `componentes_final` (todo el historico real disponible, sin corte): para el reentrenamiento
  final y el pronostico de `periodo_objetivo` -- aca no hay leakage posible porque no existe
  ningun dato mas alla de la ultima venta observada.

Esto solo funciona si la ventana de `meses_val`/`meses_test` es mas angosta que el
`horizonte` (si no, ni con un corte por combinacion alcanza para que TODAS las filas de esa
ventana queden libres de fuga) -- se verifica en tiempo de ejecucion y frena con un error
claro si no se cumple.

**Ademas, ahora es una GRILLA**: se recorren varias combinaciones de `eemd_trials` /
`eemd_noise_width` / `eemd_max_imf` (cada una paga sus 3 descomposiciones), se comparan por
`wape_val`, y se sube a Kaggle **solo la combinacion ganadora** (no todas -- a diferencia de
las grillas de `otro_pipe`/`pipe_catedra`, aca el pedido fue "elegir y subir esa").

Libreria: `EMD-signal` (se importa como `PyEMD`). Se aplica sobre `tn` agregado por
**producto** (no cliente-producto: es el nivel en el que se entrega el submit).


## 0) Setup


In [ ]:
%pip install -q EMD-signal


In [ ]:
import json, os, shutil, subprocess, time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import matplotlib.pyplot as plt
from PyEMD import EEMD
from tqdm.auto import tqdm


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET   = resolver_bucket()
DIR_RAW  = BUCKET / "datasets"
DIR_PRE  = BUCKET / "datasets_pre"
DIR_FE   = BUCKET / "datasets_fe"
RUTA_EXP = BUCKET / "exp_otro_pipe"
for d in (DIR_PRE, DIR_FE, RUTA_EXP):
    d.mkdir(parents=True, exist_ok=True)

print(f"BUCKET   : {BUCKET}")
print(f"crudos   : {DIR_RAW}")
print(f"cache pre: {DIR_PRE}")
print(f"cache FE : {DIR_FE}")
print(f"salida   : {RUTA_EXP}")


In [ ]:
def rango_meses(desde: int, hasta: int) -> list:
    a, b = (desde // 100) * 12 + desde % 100, (hasta // 100) * 12 + hasta % 100
    return [((m - 1) // 12) * 100 + ((m - 1) % 12) + 1 for m in range(a, b + 1)]


def a_m(periodo: int) -> int:
    return (periodo // 100) * 12 + (periodo % 100)


def m_a_periodo(m: int) -> int:
    return ((m - 1) // 12) * 100 + ((m - 1) % 12) + 1


## 1) Palancas


In [ ]:
PARAM = {
    'horizonte': 2,
    'max_lags': 12,
    'ventanas_ma': (3, 6),

    # -- Grilla EEMD -- se corre TODA, cada combinacion con sus 3 descomposiciones propias.
    'eemd_trials_grid': (50, 100),
    'eemd_noise_width_grid': (0.05, 0.15),
    'eemd_max_imf_grid': (3, 5),
    'eemd_n_jobs': 1,
    'min_meses_para_eemd': 18,   # series mas cortas: sin descomponer (todo al residuo)

    # -- Particion train/val/test (mismos cortes que el resto de la sesion) --
    'meses_train': rango_meses(201701, 201905),
    'meses_val':   [201907, 201908],
    'meses_test':  [201910],
    'reentrenar_con_val_para_test': True,

    # -- Entrega -------------------------------------------------------------
    'periodo_objetivo': 202002,
    'clip_min': 0.0,
    'kaggle_competition': 'labo-iii-2026-rosario',
    # True: se sube a Kaggle SOLO la combinacion ganadora (menor wape_val), una vez
    # (marcador submit_<tag>.json evita reenviar si se corre el notebook de nuevo).
    'submit': True,

    'semilla': 102191,
    'sufijo': '',

    # 'combo:<tag>' para forzar el recalculo de una combinacion puntual aunque ya
    # tenga resultado.json (ver el tag impreso al armar GRILLA_EEMD).
    'forzar': set(),
}

H = PARAM['horizonte']
L = PARAM['max_lags']
CAT_FEATURES = ['cat1', 'cat2', 'cat3', 'brand']

GRILLA_EEMD = [(t, n, m) for t in PARAM['eemd_trials_grid']
              for n in PARAM['eemd_noise_width_grid']
              for m in PARAM['eemd_max_imf_grid']]
print(f"grilla EEMD: {len(GRILLA_EEMD)} combinaciones")
for t, n, m in GRILLA_EEMD:
    print(f"  trials{t}_noise{n}_maximf{m}")

RUTA_EXP_BASE = RUTA_EXP / "eemd_grid"
RUTA_EXP_BASE.mkdir(parents=True, exist_ok=True)
print(f"salida de la grilla: {RUTA_EXP_BASE}")


## 2) Serie `tn` por producto, densificada con ceros dentro de su vida


In [ ]:
t0 = time.time()

sell = pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t")
prod = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
          .unique(subset=["product_id"]))
oficiales = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt", separator="\t")

tot_prod = (sell.group_by(["product_id", "periodo"])
                .agg(pl.col("tn").sum().alias("tn_prod"))
                .with_columns(a_m(pl.col("periodo")).alias("m")))

vida = tot_prod.group_by("product_id").agg(
    pl.col("m").min().alias("m_nace"), pl.col("m").max().alias("m_muere"))

# grilla SOLO dentro de la vida de cada producto (no antes de nacer, no despues de la
# ultima venta observada) -- ceros artificiales fuera de esa ventana meterian saltos que
# no existieron y le arruinarian la descomposicion al EMD.
grilla = (vida.with_columns(pl.int_ranges("m_nace", pl.col("m_muere") + 1).alias("m"))
              .explode("m").select("product_id", "m"))

tot_prod = (grilla.join(tot_prod.drop("periodo"), on=["product_id", "m"], how="left")
                  .with_columns(pl.col("tn_prod").fill_null(0.0))
                  .join(vida, on="product_id", how="left")
                  .join(prod.select("product_id", "cat1", "cat2", "cat3", "brand"),
                        on="product_id", how="left")
                  .with_columns(pl.col("m").map_elements(m_a_periodo, return_dtype=pl.Int64)
                                  .alias("periodo"))
                  .sort(["product_id", "m"]))

largo = vida.with_columns((pl.col("m_muere") - pl.col("m_nace") + 1).alias("n"))
print(f"productos: {tot_prod['product_id'].n_unique()}   filas: {tot_prod.height:,}   "
     f"[{time.time() - t0:.0f}s]")
print(f"largo de serie por producto: min {largo['n'].min()}  mediana {int(largo['n'].median())}  "
     f"max {largo['n'].max()}")


## 3) Cortes causales de la descomposicion (VAL/TEST)

`CORTE_VAL`/`CORTE_TEST` son el ultimo mes que puede "ver" cada descomposicion de
diagnostico: hasta ahi llega (para poder construir features en el propio mes de cada fila de
VAL/TEST), pero nunca hasta el mes objetivo de esas filas (`mes + horizonte`). Eso solo es
posible si la ventana de `meses_val`/`meses_test` es mas angosta que `horizonte` -- se
verifica explicitamente.


In [ ]:
MESES_TRAIN = sorted(a_m(p) for p in PARAM['meses_train'])
MESES_VAL   = sorted(a_m(p) for p in PARAM['meses_val'])
MESES_TEST  = sorted(a_m(p) for p in PARAM['meses_test'])

periodos_disponibles_m = set(tot_prod['m'].unique().to_list())
MESES_TRAIN = [m for m in MESES_TRAIN if m in periodos_disponibles_m]
MESES_VAL   = [m for m in MESES_VAL if m in periodos_disponibles_m]
MESES_TEST  = [m for m in MESES_TEST if m in periodos_disponibles_m]
for nombre, ms in (('TRAIN', MESES_TRAIN), ('VAL', MESES_VAL), ('TEST', MESES_TEST)):
    if not ms:
        raise ValueError(f"{nombre} quedo vacio dentro de los periodos disponibles.")
if not (max(MESES_TRAIN) < min(MESES_VAL) < max(MESES_VAL) < min(MESES_TEST)):
    raise RuntimeError("El orden cronologico train < val < test no se cumple.")

CORTE_VAL = max(MESES_VAL)
CORTE_TEST = max(MESES_TEST)

if min(MESES_VAL) + H <= CORTE_VAL:
    raise RuntimeError(
        f"meses_val abarca {CORTE_VAL - min(MESES_VAL)} meses, >= horizonte {H} -- no hay un "
        f"corte unico de EEMD que sirva para todas las filas de VAL sin fuga (alguna fila "
        f"tendria su propio mes objetivo dentro del corte). Achica meses_val o subi horizonte."
    )
if min(MESES_TEST) + H <= CORTE_TEST:
    raise RuntimeError(
        f"meses_test abarca {CORTE_TEST - min(MESES_TEST)} meses, >= horizonte {H} -- mismo "
        f"problema que con meses_val."
    )

print(f"TRAIN {len(MESES_TRAIN)} meses ({min(MESES_TRAIN)}..{max(MESES_TRAIN)})")
print(f"VAL   {MESES_VAL}   corte EEMD (val, causal):  m <= {CORTE_VAL}")
print(f"TEST  {MESES_TEST}   corte EEMD (test, causal): m <= {CORTE_TEST}")
print("chequeo de causalidad de los cortes: OK (ninguna fila de VAL/TEST puede ver su propio target)")


## 4) Descomposicion EEMD (parametrizada + cacheada por corte y por combinacion)


In [ ]:
def descomponer_uno(valores, trials, noise_width, max_imf, semilla):
    """Devuelve (max_imf+1, n) = [imf1..imfK, residuo]. El residuo se define por resta
    (residuo = valores - suma(IMFs)) -- la suma de las filas reconstruye 'valores' exacto sin
    importar cuantas IMFs haya encontrado el EMD (PyEMD NO garantiza que
    sum(IMFs devueltas) == serie original)."""
    n = len(valores)
    salida = np.zeros((max_imf + 1, n), dtype=np.float64)
    if n < PARAM['min_meses_para_eemd']:
        salida[-1] = valores
        return salida
    eemd = EEMD(trials=trials, noise_width=noise_width,
               parallel=PARAM['eemd_n_jobs'] > 1,
               processes=PARAM['eemd_n_jobs'] if PARAM['eemd_n_jobs'] > 1 else None)
    eemd.noise_seed(semilla)
    imfs = eemd.eemd(valores.astype(np.float64), max_imf=max_imf)
    k = min(imfs.shape[0], max_imf)
    salida[:k] = imfs[:k]
    salida[-1] = valores - salida[:-1].sum(axis=0)
    return salida


def descomponer_productos(tot_prod_df, trials, noise_width, max_imf, cutoff_m, tag_cutoff):
    """Descompone TODOS los productos, usando solo filas con m <= cutoff_m (None = todo el
    historico). El cache incluye el corte en el nombre, asi val/test/final -- y cada
    combinacion de la grilla -- nunca se pisan entre si."""
    componentes = [f'imf{k+1}' for k in range(max_imf)] + ['residuo']

    _tag = f'eemd_trials{trials}_noise{noise_width}_maximf{max_imf}_{tag_cutoff}'
    path_eemd = DIR_FE / f'{_tag}_componentes_otroPipe.parquet'
    if path_eemd.exists():
        return pl.read_parquet(path_eemd), componentes

    fuente = tot_prod_df if cutoff_m is None else tot_prod_df.filter(pl.col('m') <= cutoff_m)
    t0 = time.time()
    filas = []
    n_prod = fuente['product_id'].n_unique()
    for (pid,), g in tqdm(fuente.sort(['product_id', 'm']).group_by('product_id', maintain_order=True),
                          total=n_prod, desc=f'EEMD {tag_cutoff}', leave=False):
        ms = g['m'].to_list()
        comp = descomponer_uno(g['tn_prod'].to_numpy(), trials, noise_width, max_imf, PARAM['semilla'])
        for j, nombre in enumerate(componentes):
            filas.append(pl.DataFrame({'product_id': [pid] * len(ms), 'm': ms,
                                       'componente': [nombre] * len(ms), 'valor': comp[j].tolist()}))
    componentes_df = (pl.concat(filas) if filas else
                      pl.DataFrame(schema={'product_id': pl.Int64, 'm': pl.Int64,
                                          'componente': pl.Utf8, 'valor': pl.Float64}))

    # round-trip: suma(componentes) == tn_prod, siempre (no solo en una muestra).
    chk_wide = componentes_df.pivot(on='componente', index=['product_id', 'm'], values='valor')
    chk_wide = chk_wide.with_columns(pl.sum_horizontal(componentes).alias('_suma'))
    chk = chk_wide.join(fuente.select('product_id', 'm', 'tn_prod'), on=['product_id', 'm'], how='inner')
    error_max = float((chk['_suma'] - chk['tn_prod']).abs().max()) if chk.height else 0.0
    assert error_max < 1e-6, f'{tag_cutoff}: suma(componentes) != tn_prod (error max {error_max:.2e})'

    componentes_df.write_parquet(path_eemd)
    print(f'  [{tag_cutoff}] {n_prod} productos, round-trip OK (error {error_max:.1e}).   [{time.time()-t0:.0f}s]')
    return componentes_df, componentes


print('descomponer_productos() listo')


## 5) Panel de features por componente (lags, medias moviles)


In [ ]:
def armar_panel(componentes_df, componentes, tot_prod_ref):
    panel = (componentes_df.join(
                tot_prod_ref.select('product_id', 'm', 'm_nace', 'm_muere', 'cat1', 'cat2', 'cat3', 'brand'),
                on=['product_id', 'm'], how='left')
                          .sort(['product_id', 'componente', 'm']))
    panel = panel.with_columns(
        *[pl.col('valor').shift(k).over(['product_id', 'componente']).alias(f'valor_lag{k}')
         for k in range(1, L + 1)],
        *[pl.col('valor').rolling_mean(w).over(['product_id', 'componente']).alias(f'valor_ma{w}')
         for w in PARAM['ventanas_ma']],
        (pl.col('m') - pl.col('m_nace')).alias('edad_producto'),
        ((pl.col('m') - 1) % 12 + 1).alias('mes_del_anio'),
        pl.col('valor').shift(-H).over(['product_id', 'componente']).alias('y'),
        (pl.col('m') + H).map_elements(m_a_periodo, return_dtype=pl.Int64).alias('periodo_objetivo'),
    )
    panel = panel.with_columns(
        (pl.col('valor') - pl.col(f'valor_ma{PARAM["ventanas_ma"][0]}')).alias('valor_desvio_ma'))
    for c in CAT_FEATURES:
        panel = panel.with_columns(pl.col(c).cast(pl.Utf8).fill_null('NA').cast(pl.Categorical))
    return panel


print('armar_panel() listo')


## 6) WAPE + naive baseline (no dependen de la grilla, se calculan una vez)


In [ ]:
def wape(y_real, y_pred, product_ids=None, por_producto=True) -> float:
    yr = np.asarray(y_real, dtype=np.float64)
    yp = np.maximum(np.asarray(y_pred, dtype=np.float64), 0.0)
    if por_producto and product_ids is not None:
        _, inv = np.unique(np.asarray(product_ids), return_inverse=True)
        yr = np.bincount(inv, weights=yr)
        yp = np.bincount(inv, weights=yp)
    den = np.abs(yr).sum()
    return float('nan') if den == 0 else float(np.abs(yr - yp).sum() / den)


def wape_naive(meses_ev):
    """Baseline: pronostica tn_prod(t+H) = media movil de 3 meses de tn_prod en t."""
    real = tot_prod.select('product_id', pl.col('m').alias('m_obj'), 'tn_prod')
    naive = (tot_prod.select('product_id', 'm',
                             pl.col('tn_prod').rolling_mean(3).over('product_id').alias('tn_pred'))
                    .filter(pl.col('m').is_in(meses_ev))
                    .with_columns((pl.col('m') + H).alias('m_obj')))
    comparado = naive.join(real, on=['product_id', 'm_obj'], how='inner')
    if comparado.height == 0:
        return float('nan')
    return wape(comparado['tn_prod'].to_numpy(), comparado['tn_pred'].fill_null(0.0).to_numpy(),
               comparado['product_id'].to_numpy())


PARAMS_LGBM_BASE = dict(objective='regression_l1', metric='mae', verbosity=-1,
                        n_estimators=400, learning_rate=0.05, num_leaves=31,
                        min_child_samples=20, seed=PARAM['semilla'], n_jobs=-1,
                        deterministic=True, force_row_wise=True)

NAIVE_VAL = wape_naive(MESES_VAL)
NAIVE_TEST = wape_naive(MESES_TEST)
print(f'naive (ma3): val={NAIVE_VAL:.5f}   test={NAIVE_TEST:.5f}')


## 7) `correr_combo()` — 3 descomposiciones + 3 rondas de entrenamiento + submit


In [ ]:
def entrenar_componente(panel_x, componente, meses_fit):
    features_x = [c for c in panel_x.columns if c not in
                 {'product_id', 'componente', 'm', 'm_nace', 'm_muere', 'periodo_objetivo', 'y'}]
    cat_en_features = [c for c in CAT_FEATURES if c in features_x]
    b = (panel_x.filter((pl.col('componente') == componente) & pl.col('m').is_in(meses_fit)
                        & pl.col('y').is_not_null())
               .to_pandas())
    modelo = lgb.LGBMRegressor(**PARAMS_LGBM_BASE)
    modelo.fit(b[features_x], b['y'], categorical_feature=cat_en_features)
    return modelo, features_x


def predecir_componente(modelo, features_x, bloque_pl):
    return modelo.predict(bloque_pl.to_pandas()[features_x])


def evaluar(modelos, features_por_comp, panel_x, componentes_x, meses_ev):
    filas = []
    for comp in componentes_x:
        b = panel_x.filter((pl.col('componente') == comp) & pl.col('m').is_in(meses_ev))
        if b.height == 0:
            continue
        pred = predecir_componente(modelos[comp], features_por_comp[comp], b)
        filas.append(b.select('product_id', 'm').with_columns(pl.Series('pred', pred)))
    if not filas:
        return float('nan')
    pred_total = (pl.concat(filas).group_by(['product_id', 'm'])
                        .agg(pl.col('pred').sum().alias('tn_pred'))
                        .with_columns((pl.col('m') + H).alias('m_obj')))
    real = tot_prod.select('product_id', pl.col('m').alias('m_obj'), 'tn_prod')
    comparado = pred_total.join(real, on=['product_id', 'm_obj'], how='inner')
    return wape(comparado['tn_prod'].to_numpy(), np.maximum(comparado['tn_pred'].to_numpy(), PARAM['clip_min']),
               comparado['product_id'].to_numpy())


def correr_combo(trials, noise_width, max_imf):
    tag = f'trials{trials}_noise{noise_width}_maximf{max_imf}'
    dir_out = RUTA_EXP_BASE / tag
    dir_out.mkdir(parents=True, exist_ok=True)
    path_resultado = dir_out / 'resultado.json'

    if path_resultado.exists() and f'combo:{tag}' not in PARAM['forzar']:
        print(f'[{tag}] cache encontrada -> se saltea')
        with open(path_resultado) as f:
            return json.load(f)

    print(f"\n{'='*70}\n{tag}\n{'='*70}")
    t0 = time.time()

    componentes_df_val, componentes_x = descomponer_productos(
        tot_prod, trials, noise_width, max_imf, CORTE_VAL, f'{tag}_corteVal{CORTE_VAL}')
    componentes_df_test, _ = descomponer_productos(
        tot_prod, trials, noise_width, max_imf, CORTE_TEST, f'{tag}_corteTest{CORTE_TEST}')
    componentes_df_final, _ = descomponer_productos(
        tot_prod, trials, noise_width, max_imf, None, f'{tag}_full')

    panel_val = armar_panel(componentes_df_val, componentes_x, tot_prod)
    panel_test = armar_panel(componentes_df_test, componentes_x, tot_prod)
    panel_final = armar_panel(componentes_df_final, componentes_x, tot_prod)

    modelos_val, feats_val = {}, {}
    for comp in componentes_x:
        modelos_val[comp], feats_val[comp] = entrenar_componente(panel_val, comp, MESES_TRAIN)
    wape_val = evaluar(modelos_val, feats_val, panel_val, componentes_x, MESES_VAL)

    meses_fit_test = (MESES_TRAIN + MESES_VAL) if PARAM['reentrenar_con_val_para_test'] else MESES_TRAIN
    modelos_test, feats_test = {}, {}
    for comp in componentes_x:
        modelos_test[comp], feats_test[comp] = entrenar_componente(panel_test, comp, meses_fit_test)
    wape_test = evaluar(modelos_test, feats_test, panel_test, componentes_x, MESES_TEST)

    print(f'  wape_val={wape_val:.5f} (naive {NAIVE_VAL:.5f})   wape_test={wape_test:.5f} (naive {NAIVE_TEST:.5f})')

    # Entrenamiento final + submit: usa la descomposicion FULL (todo el historico real --
    # no hay fuga posible, no existe nada mas alla de la ultima venta observada).
    sup_final = panel_final.filter(pl.col('y').is_not_null())
    meses_fit_final = sorted(sup_final['m'].unique().to_list())
    modelos_final, feats_final = {}, {}
    for comp in componentes_x:
        modelos_final[comp], feats_final[comp] = entrenar_componente(panel_final, comp, meses_fit_final)

    infer = panel_final.filter(pl.col('y').is_null())
    meses_infer = sorted(infer['m'].unique().to_list())[-H:]
    infer = infer.filter(pl.col('m').is_in(meses_infer))

    filas_pred = []
    for comp in componentes_x:
        b = infer.filter(pl.col('componente') == comp)
        if b.height == 0:
            continue
        pred = predecir_componente(modelos_final[comp], feats_final[comp], b)
        filas_pred.append(b.select('product_id', 'periodo_objetivo').with_columns(pl.Series('pred', pred)))
    pred_final = (pl.concat(filas_pred).group_by(['product_id', 'periodo_objetivo'])
                        .agg(pl.col('pred').sum().alias('tn')))

    OBJ = PARAM['periodo_objetivo']
    obj = pred_final.filter(pl.col('periodo_objetivo') == OBJ)
    submit = oficiales.select('product_id').join(obj.select('product_id', 'tn'), on='product_id', how='left')
    sin_pred = int(submit['tn'].null_count())
    submit = submit.with_columns(pl.col('tn').fill_null(0.0).clip(lower_bound=PARAM['clip_min'])).sort('product_id')

    path_submit = dir_out / f'submission_{OBJ}.csv'
    submit.write_csv(path_submit)

    resultado = {
        'tag': tag, 'eemd_trials': trials, 'eemd_noise_width': noise_width, 'eemd_max_imf': max_imf,
        'periodo_objetivo': OBJ, 'wape_val': wape_val, 'wape_test': wape_test,
        'wape_naive_val': NAIVE_VAL, 'wape_naive_test': NAIVE_TEST,
        'corte_val': CORTE_VAL, 'corte_test': CORTE_TEST,
        'sin_prediccion': sin_pred, 'tn_total_entregado': float(submit['tn'].sum()),
        'path_submit': str(path_submit.relative_to(BUCKET)),
    }
    with open(path_resultado, 'w', encoding='utf-8') as f:
        json.dump(resultado, f, indent=2, ensure_ascii=False, default=str)
    print(f'  [{tag}] listo.   [{time.time()-t0:.0f}s]')
    return resultado


print('correr_combo() listo')


## 8) Correr TODA la grilla (resumible) + tabla comparativa


In [ ]:
t0 = time.time()
resultados = [correr_combo(t, n, m) for (t, n, m) in GRILLA_EEMD]
print(f"\n{'='*70}\n{len(resultados)}/{len(GRILLA_EEMD)} combinaciones completas.   [{time.time()-t0:.0f}s]\n{'='*70}")

tabla = pl.DataFrame(resultados).select(
    ['tag', 'eemd_trials', 'eemd_noise_width', 'eemd_max_imf', 'wape_val', 'wape_test']
).sort('wape_val')
print(tabla)

path_tabla = RUTA_EXP_BASE / 'grilla_eemd.csv'
tabla.write_csv(path_tabla)
print(f'\nTabla guardada: {path_tabla}')

ganador = min(resultados, key=lambda r: r['wape_val'])
print(f"\nGanador (menor wape_val): {ganador['tag']}")
print(f"  wape_val={ganador['wape_val']:.5f}  (naive {ganador['wape_naive_val']:.5f})")
print(f"  wape_test={ganador['wape_test']:.5f}  (naive {ganador['wape_naive_test']:.5f})")


### Ejemplo visual del ganador (repasar a ojo antes de subir)

Recarga la descomposicion FULL del combo ganador (ya esta cacheada, no vuelve a calcular
nada) y grafica los 2 productos de mayor volumen.


In [ ]:
_g = ganador
componentes_df_ganador, componentes_ganador = descomponer_productos(
    tot_prod, _g['eemd_trials'], _g['eemd_noise_width'], _g['eemd_max_imf'],
    None, f"trials{_g['eemd_trials']}_noise{_g['eemd_noise_width']}_maximf{_g['eemd_max_imf']}_full")

SERIE = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7']
TINTA, GRILLA_C = '#0b0b0b', '#e1e0d9'
plt.rcParams.update({'axes.grid': True, 'grid.color': GRILLA_C, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 9, 'figure.dpi': 110})

top_vol = (tot_prod.group_by('product_id').agg(pl.col('tn_prod').sum().alias('t'))
                  .sort('t', descending=True).head(2)['product_id'].to_list())

for pid in top_vol:
    wide = (componentes_df_ganador.filter(pl.col('product_id') == pid)
                                  .pivot(on='componente', index='m', values='valor').sort('m'))
    orig = tot_prod.filter(pl.col('product_id') == pid).sort('m')['tn_prod']
    n_comp = len(componentes_ganador)
    fig, axes = plt.subplots(n_comp + 1, 1, figsize=(8, 1.3 * (n_comp + 1)), sharex=True)
    axes[0].plot(wide['m'], orig, color=TINTA, linewidth=1.4)
    axes[0].set_title(f'producto {pid} -- tn_prod original ({_g["tag"]})', loc='left', fontsize=8)
    for j, nombre in enumerate(componentes_ganador, start=1):
        axes[j].plot(wide['m'], wide[nombre], color=SERIE[(j - 1) % len(SERIE)], linewidth=1.1)
        axes[j].set_title(nombre, loc='left', fontsize=8)
    fig.tight_layout()
    plt.show()


## 9) Submit del ganador a Kaggle (opcional, solo esa combinacion)


In [ ]:
def kaggle_cli(args):
    try:
        r = subprocess.run(['kaggle'] + args, capture_output=True, text=True)
        return r.returncode == 0, (r.stdout or '') + (r.stderr or '')
    except FileNotFoundError:
        return False, 'La CLI de kaggle no esta instalada.  pip install kaggle'
    except Exception as e:
        return False, f'{type(e).__name__}: {e}'


path_marca_submit = RUTA_EXP_BASE / f"submit_{ganador['tag']}.json"

if not PARAM['submit']:
    print("PARAM['submit'] = False -> no se sube nada. Los CSV de las " + str(len(GRILLA_EEMD)) + " combinaciones ya estan generados.")
elif path_marca_submit.exists():
    print(f'ya se subio antes ({path_marca_submit.name} existe) -> no se reenvia.')
else:
    kdst = Path.home() / '.kaggle' / 'kaggle.json'
    kdst.parent.mkdir(parents=True, exist_ok=True)
    if not kdst.exists():
        for cand in (BUCKET / 'kaggle.json', BUCKET / 'kaggle' / 'kaggle.json'):
            if cand.exists():
                shutil.copy(cand, kdst); kdst.chmod(0o600)
                break
    if not kdst.exists():
        print('Sin credenciales de Kaggle: no se sube. El CSV ya esta generado.')
    else:
        kdst.chmod(0o600)
        path_submit_ganador = BUCKET / ganador['path_submit']
        msg = (f"EEMD grid ganador {ganador['tag']} | wape_val={ganador['wape_val']:.5f} "
              f"wape_test={ganador['wape_test']:.5f}")
        ok, salida = kaggle_cli(['competitions', 'submit', '-c', PARAM['kaggle_competition'],
                                '-f', str(path_submit_ganador), '-m', msg[:200]])
        print(f'mensaje: {msg}\n{salida}')
        if ok:
            with open(path_marca_submit, 'w', encoding='utf-8') as f:
                json.dump({'ganador': ganador['tag'], 'mensaje': msg}, f, indent=2, ensure_ascii=False)
            print('Submit del ganador enviado a Kaggle.')
        else:
            print('NO se pudo subir; el CSV esta en disco.')


Repasar a ojo:
  - [ ] en los graficos de la seccion 8, la suma visual de IMFs + residuo del ganador se
    parece a la serie original (`tn_prod`)?
  - [ ] el residuo (ultimo panel) se ve como una tendencia suave, sin los saltos mes a mes
    que si tiene `tn_prod`?
  - [ ] las IMFs de mayor frecuencia oscilan alrededor de cero, sin arrastrar un nivel?
  - [ ] el `wape_val`/`wape_test` del ganador le gana al naive (`ma3`)? si ninguna
    combinacion de la grilla le gana, la descomposicion no esta aportando sobre simplemente
    seguir la tendencia reciente.
  - [ ] `tabla` (seccion 8) -- hay una combinacion claramente mejor, o estan todas muy
    parecidas (en cuyo caso el resultado depende mas de la semilla que de los parametros)?
